In [ ]:
import requests
import tkinter as tk

AUTHOR_SEARCH_URL = "https://openlibrary.org/search/authors.json"
BOOK_SEARCH_URL = "https://openlibrary.org/search.json"

# Search authors by name
def search_authors_api(keyword: str):
    params = {"q": keyword}
    response = requests.get(AUTHOR_SEARCH_URL, params=params)
    if response.status_code == 200:
        return response.json()
    return None

# Search books by author name
def get_books_by_author_name(author_name):
    params = {"author": author_name}
    response = requests.get(BOOK_SEARCH_URL, params=params)
    if response.status_code == 200:
        return response.json().get("docs", [])
    return []

# Extract author info
def extract_author_info(author):
    name = author.get("name", "Unknown Name")
    top_work = author.get("top_work", "N/A")
    ratings = author.get("ratings_average", "N/A")
    return name, top_work, ratings

# GUI class
class LibraryAppGUI:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Library App")

        self.label = tk.Label(self.root, text="Library App", font=('Arial', 18, 'bold'))
        self.label.grid(row=0, column=0, padx=10, pady=10)

        self.entry = tk.Entry(self.root, width=40, font=('Arial', 12))
        self.entry.grid(row=0, column=1, padx=10, pady=10)

        self.button_author = tk.Button(
            self.root,
            text="SEARCH AUTHOR",
            font=('Arial', 10, "bold"),
            command=self.search_author
        )
        self.button_author.grid(row=0, column=2, padx=2, pady=4)

        self.output_box = tk.Text(self.root, width=80, height=25, wrap="word")
        self.output_box.tag_config("primary_heading", font=('Arial', 24, "bold"))
        self.output_box.tag_config("subheading", font=('Arial', 14, "bold"))
        self.output_box.tag_config("normal", font=('Arial', 12))
        self.output_box.grid(row=1, column=0, columnspan=3, padx=10, pady=10)

        self.root.mainloop()

    def search_author(self):
        self.output_box.delete("1.0", tk.END)

        keyword = self.entry.get()
        author_data = search_authors_api(keyword)

        if not author_data:
            self.output_box.insert(tk.END, "Error retrieving author data.\n", "normal")
            return

        results = author_data.get("docs", [])
        if not results:
            self.output_box.insert(tk.END, "No authors found.\n", "normal")
            return

        # Use the first author returned
        author_info = results[0]
        name, top_work, ratings = extract_author_info(author_info)

        # Display author heading
        self.output_box.insert(tk.END, f"{name}\n", "primary_heading")
        self.output_box.insert(tk.END, "------------------------------------------\n", "normal")

        # Display author info
        self.output_box.insert(tk.END, "Famous Work: ", "subheading")
        self.output_box.insert(tk.END, f"{top_work}\n", "normal")
        self.output_box.insert(tk.END, "Rating Score: ", "subheading")
        self.output_box.insert(tk.END, f"{ratings}\n\n", "normal")

        # Fetch books by author name
        books = get_books_by_author_name(name)
        self.output_box.insert(tk.END, "Other Books:\n", "subheading")
        self.output_box.insert(tk.END, "---------------------\n", "normal")

        if not books:
            self.output_box.insert(tk.END, "No books were found for this author.\n", "normal")
            return

        # Show up to 10 books
        for book in books[:10]:
            title = book.get("title", "Unknown Title")
            year = book.get("first_publish_year", "Unknown")
            self.output_box.insert(tk.END, f"• {title} ({year})\n", "normal")

        self.output_box.insert(tk.END, "\n")

# Run the app
LibraryAppGUI()


In [4]:
#RUN APPLICATION
root = tk.Tk()
app = LibraryApp(root)
root.mainloop()